# 🎓 Placify — Quick Demo

This branch sends the **entire company dataset** directly to the LLM — no pre-filtering,  
no TF-IDF shortlisting. The LLM itself reads every company and selects the best 5.

```
  INPUT: Resume PDF
       ↓
   Stage 1 — EDA & Resume Loading
       ↓
   Stage 2 — LLM Analysis  (All Companies → Gemini/Groq/Ollama → Top 5)
       ↓
   Stage 3 — Visualisations + PDF Report
```

---


## ⚙️ Cell 1 — Setup & Libraries

In [ ]:
%matplotlib inline
import os, sys, json, ast, warnings, textwrap, io, contextlib, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, IFrame
from pypdf import PdfReader
from fpdf import FPDF

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'axes.titlesize': 14, 'axes.titleweight': 'bold', 'figure.dpi': 110})
FIG_W, FIG_H = 12, 5

print('✅ Libraries loaded.')


## 🔑 Cell 2 — Paths & Environment
All folders are defined here. Environment variables are loaded by `ai_service.py` automatically.

In [ ]:
BASE_DIR    = Path('.').resolve()
COMPANY_DIR = BASE_DIR / 'company_dataset'
RESUME_DIR  = BASE_DIR / 'web_data' / 'resume'
REPORT_DIR  = BASE_DIR / 'web_data' / 'pdf'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Paths configured.')
print('   Companies :', COMPANY_DIR)
print('   Resumes   :', RESUME_DIR)
print('   Reports   :', REPORT_DIR)


## 📂 Cell 3 — Load Company Dataset
Read all companies from `companies.json`. These will be sent **in full** to the LLM.

In [ ]:
companies_df  = pd.DataFrame()
companies_raw = []

company_file = COMPANY_DIR / 'companies.json'
if not company_file.exists():
    print('❌ companies.json not found.')
else:
    companies_raw = json.loads(company_file.read_text(encoding='utf-8'))
    if not isinstance(companies_raw, list):
        companies_raw = [companies_raw]
    companies_df = pd.DataFrame(companies_raw)

    print(f'✅ Loaded {len(companies_df)} companies.')
    print(f'   Columns : {list(companies_df.columns)}')
    print()

    # Clean display — join skill lists into comma strings
    preview = companies_df[['name', 'role', 'location', 'skills']].head(5).copy()
    preview['skills'] = preview['skills'].apply(
        lambda s: ', '.join(s) if isinstance(s, list) else str(s)
    )
    pd.set_option('display.max_colwidth', 60)
    display(preview)


## 📊 Cell 4 — Exploratory Data Analysis
Three charts to understand the dataset before the LLM processes it.

In [ ]:
if companies_df.empty:
    print('⚠️ No company data — run Cell 3 first.')
else:
    # ── Chart 1: Top 10 Skills ───────────────────────────────────────────
    all_skills = []
    for s in companies_df.get('skills', pd.Series([])).dropna():
        if isinstance(s, list): all_skills.extend([x.strip() for x in s])
        elif isinstance(s, str):
            try: all_skills.extend(ast.literal_eval(s))
            except: all_skills.extend([x.strip() for x in s.split(',')])
    skill_cnt = Counter(all_skills).most_common(10)
    if skill_cnt:
        sk, sv = zip(*skill_cnt)
        fig1, ax1 = plt.subplots(figsize=(FIG_W, FIG_H))
        sns.barplot(ax=ax1, y=list(sk), x=list(sv), hue=list(sk), palette='viridis', legend=False)
        ax1.set_title('🔧 Top 10 Most In-Demand Skills')
        ax1.set_xlabel('Number of Companies')
        for i, v in enumerate(sv): ax1.text(v + 0.1, i, str(v), va='center', fontsize=9)
        plt.tight_layout(); display(fig1); plt.close(fig1)
        plt.show()

    # ── Chart 2: Top 10 Roles ────────────────────────────────────────────
    if 'role' in companies_df.columns:
        rc = companies_df['role'].dropna().value_counts().head(10)
        fig2, ax2 = plt.subplots(figsize=(FIG_W, FIG_H))
        sns.barplot(ax=ax2, y=rc.index.tolist(), x=rc.values.tolist(), hue=rc.index.tolist(), palette='crest', legend=False)
        ax2.set_title('💼 Top 10 Most Common Roles')
        ax2.set_xlabel('Count')
        for i, v in enumerate(rc.values): ax2.text(v + 0.1, i, str(v), va='center', fontsize=9)
        plt.tight_layout(); display(fig2); plt.close(fig2)
        plt.show()

    # ── Chart 3: Location Distribution ───────────────────────────────────
    if 'location' in companies_df.columns:
        lc = companies_df['location'].fillna('Unknown').value_counts().head(8)
        fig3, ax3 = plt.subplots(figsize=(FIG_W, FIG_H))
        sns.barplot(ax=ax3, y=lc.index.tolist(), x=lc.values.tolist(), hue=lc.index.tolist(), palette='rocket', legend=False)
        ax3.set_title('📍 Job Location Distribution')
        ax3.set_xlabel('Count')
        for i, v in enumerate(lc.values): ax3.text(v + 0.1, i, str(v), va='center', fontsize=9)
        plt.tight_layout(); display(fig3); plt.close(fig3)
        plt.show()


## 🎒 Cell 5 — Resume Selection
List all PDFs in `web_data/resume/` and choose which one to analyse.

In [ ]:
TARGET_RESUME_INDEX = 2

In [ ]:
resume_files = sorted(RESUME_DIR.glob('*.pdf')) if RESUME_DIR.exists() else []

if not resume_files:
    print('❌ No PDF resumes found in web_data/resume/')
else:
    print('📄 Available resumes:')
    for fi, f in enumerate(resume_files):
        marker = '  ◀ SELECTED' if fi == TARGET_RESUME_INDEX else ''
        print(f'   [{fi}] {f.name}{marker}')
    selected_resume = resume_files[min(TARGET_RESUME_INDEX, len(resume_files) - 1)]


## 📖 Cell 6 — Resume Text Extraction
Extract raw text from the selected PDF. A regex cleaner fixes common multi-column PDF artefacts.

In [ ]:
import re
resume_text = ''

def clean_pdf_text(raw: str) -> str:
    text = re.sub(r'(?<=[A-Za-z])\n(?=[A-Za-z])', '', raw)
    text = re.sub(r' {2,}', ' ', text)
    text = '\n'.join(l for l in text.splitlines() if l.strip())
    return text.strip()

if 'selected_resume' not in dir():
    print('❌ Run Cell 5 first.')
else:
    try:
        reader = PdfReader(str(selected_resume))
        resume_text = clean_pdf_text('\n'.join(p.extract_text() or '' for p in reader.pages))
        print(f'✅ Extracted {len(resume_text)} chars ({len(resume_text.split())} words) from {selected_resume.name}')
        print('\n─── First 600 chars ─────────────────────────────────────')
        print(resume_text[:600])
        print('─────────────────────────────────────────────────────────')
    except Exception as e:
        print(f'❌ Failed to read PDF: {e}')


## 🤖 Cell 7 — LLM Analysis (Direct Full-Database Mode)

**All** companies are serialised into JSON and sent directly to the AI service.  
The LLM reads every company and independently selects the best 5 matches.  

> Provider chain: **Gemini → Groq → Ollama** (automatic fallback)


In [ ]:
import io, contextlib
from app.services.ai_service import analyze_profile

llm_report = None

if companies_df.empty or not resume_text:
    print('❌ Run Cells 3 and 6 first.')
else:
    candidates_json = json.dumps([{
        'name'       : c.get('name', ''),
        'role'       : c.get('role', ''),
        'skills'     : c.get('skills', []),
        'email'      : c.get('email', ''),
        'description': c.get('description', ''),
        'location'   : c.get('location', '')
    } for c in companies_raw])

    user_context = (
        '--- QUICK DEMO MODE ---\n'
        'No quiz. Matching based solely on resume.\n'
        '--- RESUME CONTENT ---\n'
        + resume_text[:4000] + '...'
    )

    print(f'🚀 Sending ALL {len(companies_raw)} companies + resume to AI service...')
    print(f'   Payload size: {len(candidates_json):,} chars')
    print()

    try:
        _buf = io.StringIO()
        with contextlib.redirect_stdout(_buf):
            llm_report = analyze_profile(
                user_context     = user_context,
                candidates_json  = candidates_json,
                mode             = 'Quick Demo',
                answers          = {},
                resume_extracted = resume_text
            )

        if llm_report:
            print('✅ AI report received!')
            print('   Candidate      :', llm_report.get('candidate_name', '?'))
            print('   Readiness Score:', llm_report.get('readiness_score', '?'), '/ 100')
            print()
            recs = llm_report.get('job_recommendations', [])
            if recs:
                print('🏆 Top', len(recs), 'Job Recommendations:')
                print('─' * 65)
                rec_rows = []
                for j, rec in enumerate(recs, 1):
                    print(f'  {j}. {rec.get("company", "?")} — {rec.get("role", "?")}')
                    print(f'     📍 {rec.get("location", "?")}  |  {rec.get("match", "")[:65]}...')
                    print()
                    rec_rows.append({'#': j, 'Company': rec.get('company','?'),
                                     'Role': rec.get('role','?'), 'Location': rec.get('location','?')})
                display(pd.DataFrame(rec_rows).set_index('#'))
        else:
            print('❌ AI returned no data — check API keys in .env')
    except Exception as e:
        print('❌ AI call failed:', e)


## 📊 Cell 8 — Readiness Score & Skill Insights

In [ ]:
if not llm_report:
    print('⚠️ No LLM report — run Cell 7 first.')
else:
    score  = llm_report.get('readiness_score', 0)
    colour = '#2ecc71' if score >= 70 else ('#f39c12' if score >= 40 else '#e74c3c')
    label  = 'Ready 🟢' if score >= 70 else ('Developing 🟡' if score >= 40 else 'Needs Work 🔴')

    fig1, ax1 = plt.subplots(figsize=(10, 2.8))
    ax1.barh([''], [score],           color=colour,    height=0.5)
    ax1.barh([''], [100 - score], left=[score], color='#ecf0f1', height=0.5)
    ax1.text(min(score - 2, 93), 0, f'{score}/100',
             va='center', ha='right', fontsize=17, fontweight='bold', color='white')
    ax1.text(102, 0, label, va='center', fontsize=11)
    ax1.set_xlim(0, 115)
    ax1.set_title('Placement Readiness — ' + llm_report.get('candidate_name', 'Student'),
                  fontsize=13, fontweight='bold')
    ax1.yaxis.set_visible(False)
    ax1.set_xlabel('Score / 100')
    plt.tight_layout()
    display(fig1); plt.close(fig1)

    # ── Insight Cards ─────────────────────────────────────────────────
    strengths = llm_report.get('strengths', [])
    gaps      = llm_report.get('gaps', [])
    acts      = llm_report.get('action_plan', [])
    max_items = max(len(strengths), len(gaps), len(acts), 1)
    ROW_H, HEADER = 1.5, 1.2
    fig2, axes = plt.subplots(1, 3, figsize=(18, HEADER + max_items * ROW_H + 0.5))
    for ax, (cat, items, bg, hcol) in zip(axes, [
        ('Strengths',   strengths, '#eafaf1', '#1e8449'),
        ('Skill Gaps',  gaps,      '#fef9e7', '#9a7d0a'),
        ('Action Plan', acts,      '#eaf4fb', '#1a5276')
    ]):
        ax.set_facecolor(bg); ax.set_xlim(0, 1)
        top = HEADER + max_items * ROW_H
        ax.set_ylim(0, top); ax.axis('off')
        ax.text(0.5, top - 0.5, cat, ha='center', va='center', fontsize=13,
                fontweight='bold', color='white',
                bbox=dict(boxstyle='round,pad=0.4', facecolor=hcol, edgecolor='none'))
        for j, item in enumerate(items):
            y_pos = top - HEADER - j * ROW_H - ROW_H * 0.5
            ax.text(0.06, y_pos, '\u2022  ' + textwrap.fill(item, 44),
                    va='center', fontsize=10, color='#1c2833', linespacing=1.4)
    plt.suptitle('Student Profile Insights — LLM Generated', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    display(fig2); plt.close(fig2)


## 📄 Cell 9 — PDF Report Generation
Generate and render the placement report inline.

In [ ]:
pdf_path = None

if not llm_report:
    print('⚠️ No LLM report — run Cell 7 first.')
else:
    class PlacifyPDF(FPDF):
        def header(self):
            self.set_fill_color(27, 54, 93)
            self.rect(0, 0, 210, 22, 'F')
            self.set_font('Helvetica', 'B', 16)
            self.set_text_color(255, 255, 255)
            self.set_y(6)
            self.cell(0, 10, 'Placify - Placement Assessment Report', align='C', ln=True)
            self.ln(12); self.set_text_color(0, 0, 0)
        def footer(self):
            self.set_y(-13)
            self.set_font('Helvetica', 'I', 8)
            self.set_text_color(150, 150, 150)
            self.cell(0, 8, 'Generated by Placify AI  |  Page ' + str(self.page_no()), align='C')
        def section_title(self, t):
            self.set_fill_color(240, 244, 255)
            self.set_font('Helvetica', 'B', 12)
            self.set_text_color(27, 54, 93)
            self.cell(0, 9, t, ln=True, fill=True)
            self.set_text_color(0, 0, 0); self.ln(2)
        def bullet_list(self, items):
            self.set_font('Helvetica', '', 10)
            for item in items: self.multi_cell(0, 6, '  -  ' + str(item))
            self.ln(3)

    pdf = PlacifyPDF(); pdf.add_page()
    score = llm_report.get('readiness_score', 0)
    pdf.set_font('Helvetica', 'B', 13)
    pdf.cell(0, 8, 'Candidate: ' + llm_report.get('candidate_name', 'Student'), ln=True)
    pdf.set_font('Helvetica', '', 11)
    pdf.cell(0, 7, 'Readiness Score: ' + str(score) + '/100', ln=True); pdf.ln(5)
    pdf.section_title('Strengths');    pdf.bullet_list(llm_report.get('strengths', []))
    pdf.section_title('Skill Gaps');   pdf.bullet_list(llm_report.get('gaps', []))
    pdf.section_title('Action Plan');  pdf.bullet_list(llm_report.get('action_plan', []))
    pdf.section_title('Top 5 Recommendations')
    for i_r, rec in enumerate(llm_report.get('job_recommendations', []), 1):
        pdf.set_font('Helvetica', 'B', 10)
        pdf.cell(0, 6, str(i_r) + '. ' + rec.get('company','?') + ' - ' + rec.get('role','?'), ln=True)
        pdf.set_font('Helvetica', '', 10)
        pdf.multi_cell(0, 5.5, '   Location: ' + rec.get('location','?') + '\n   Match   : ' + rec.get('match','?'))
        pdf.ln(3)

    filename = 'Placement_Report_' + str(int(time.time())) + '.pdf'
    pdf_path  = str(REPORT_DIR / filename)
    pdf.output(pdf_path)
    print('✅ PDF saved to: web_data/pdf/' + filename)
    display(IFrame('web_data/pdf/' + filename, width=900, height=640))


## ✉️ Cell 10 — Email Draft Viewer
Change `EMAIL_DRAFT_INDEX` (0–4) to preview the cold email for that recommendation.

In [ ]:
EMAIL_DRAFT_INDEX = 2

In [ ]:
if not llm_report:
    print('⚠️ No LLM report — run Cell 7 first.')
else:
    recs = llm_report.get('job_recommendations', [])
    if not recs:
        print('⚠️ No recommendations in the report.')
    else:
        idx   = min(EMAIL_DRAFT_INDEX, len(recs) - 1)
        rec   = recs[idx]
        draft = rec.get('email_draft', 'No email draft available.')
        print('✉️  Email Draft — Recommendation #' + str(idx + 1))
        print('   Company  :', rec.get('company', '?'))
        print('   Role     :', rec.get('role', '?'))
        print('   Location :', rec.get('location', '?'))
        print('\n' + '─' * 65)
        print(draft)
        print('─' * 65)


---
## ✅ Demo Complete!

| Cell | What it does |
|------|--------------|
| 1–2  | Setup + Paths |
| 3–4  | Load {len(companies)} companies + EDA charts |
| 5–6  | Select resume + Extract text |
| **7**| **🤖 Direct LLM — all companies sent, Top 5 selected** |
| 8    | Readiness gauge + Insight cards |
| 9    | PDF report generated |
| 10   | View email draft by index |

> Change `TARGET_RESUME_INDEX` in Cell 5 to switch students.
